# Графы в DGL

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Курс "Машинное обучение на графах", Лекции 4-5 "Графовые нейронные сети"
* Документация:
    * https://docs.dgl.ai/tutorials/blitz/2_dglgraph.html
    * https://docs.dgl.ai/guide/graph.html
    * https://docs.dgl.ai/api/python/dgl.data.html
    * https://docs.dgl.ai/en/latest/generated/dgl.DGLGraph.num_nodes.html
    * https://docs.dgl.ai/generated/dgl.from_networkx.html
    * https://docs.dgl.ai/generated/dgl.heterograph.html

## Вопросы для совместного обсуждения

1\.  Обсудите основные возможности по созданию графов и работы с графами в `dgl`.

In [1]:
%pip uninstall numpy dgl torch -y
%pip install "numpy<2" "dgl>=2.0" "torch>=2.0"
%pip install torchdata==0.6.0

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Cоздайте граф "путь", состоящий из 5 вершин, и представьте его в виде `dgl.graph`.

Создайте двумерный тензор размера 5х2 при помощи функции `torch.rand` и сохраните его в качестве атрибутов узлов `features`. Аналогичным образом создайте атрибут ребер `weight`. Выведите основные характеристики графа: количество узлов, количество ребер, размерность признаков узлов и ребер.

Выведите входящую и исходящую степень узлов.

- [ ] Проверено на семинаре

In [1]:
import torch
import dgl

In [2]:
# создаём путь из 5 вершин: рёбра 0→1, 1→2, 2→3, 3→4
src = torch.tensor([0, 1, 2, 3])
dst = torch.tensor([1, 2, 3, 4])
g = dgl.graph((src, dst))

In [3]:
# атрибуты вершин: тензор 5×2
g.ndata['features'] = torch.rand(5, 2)

# атрибуты рёбер: тензор 4×2
g.edata['weight'] = torch.rand(g.num_edges(), 2)

In [4]:
# выводим характеристики
print("Number of nodes:", g.num_nodes())
print("Number of edges:", g.num_edges())
print("Node feature shape:", g.ndata['features'].shape)
print("Edge weight shape:", g.edata['weight'].shape)


Number of nodes: 5
Number of edges: 4
Node feature shape: torch.Size([5, 2])
Edge weight shape: torch.Size([4, 2])


In [5]:
# степени вершин
print("In-degrees:", g.in_degrees().tolist())
print("Out-degrees:", g.out_degrees().tolist())


In-degrees: [0, 1, 1, 1, 1]
Out-degrees: [1, 1, 1, 1, 0]


<p class="task" id="2"></p>

2\. Загрузите граф карате клуба из `networkx`. Закодируйте значения атрибута `club` на узлах числами 0 (для клуба "Mr. Hi") и 1 (для клуба "Officer").

Преобразуйте его к графу `dgl.graph` при помощи `dgl.from_networkx`, сохранив значения атрибута узлов `club` и значения атрибутов ребер `weight`.

Добавьте в качестве атрибутов узлов единичную матрицу размера 34х34. Выведите основные характеристики графа: количество узлов, количество ребер, размерность признаков узлов, кол-во уникальных классов узлов.

Создайте новую версию графа с добавленными петлями при помощи метода `add_self_loop`. Выведите на экран количество ребер в новом графе. Выведите на экран идентификаторы ребер-петель, найденные двумя способами:
* получите тензоры начала и конца ребер, найдите индексы, для которых элементы двух тензоров совпадают;
* создайте набор идентификаторов узлов при помощи `torch.arange` и найдите петли при помощи метода `edge_ids`.


- [ ] Проверено на семинаре

In [6]:
import networkx as nx

In [7]:
nx_g = nx.karate_club_graph()
for u, v in nx_g.edges():
    nx_g[u][v]['weight'] = 1.0

In [8]:
club_map = {'Mr. Hi': 0, 'Officer': 1}
for n, data in nx_g.nodes(data=True):
    data['club'] = club_map[data['club']]

In [9]:
# Конвертим в ориентированный граф
nx_dg = nx.DiGraph(nx_g)

# Преобразуем в DGL, сохранив 'club' и 'weight'
g = dgl.from_networkx(nx_dg, node_attrs=['club'], edge_attrs=['weight'])


In [10]:
g.ndata['id'] = torch.eye(g.num_nodes())

In [11]:
print("Nodes:", g.num_nodes())
print("Edges:", g.num_edges())
print("Node feat shapes:", {k: v.shape for k, v in g.ndata.items()})
print("Unique classes:", len(torch.unique(g.ndata['club'])))

Nodes: 34
Edges: 156
Node feat shapes: {'club': torch.Size([34]), 'id': torch.Size([34, 34])}
Unique classes: 2


In [12]:
g2 = dgl.add_self_loop(g)
print("Edges with self‐loops:", g2.num_edges())

Edges with self‐loops: 190


In [13]:
src, dst = g2.edges()

In [14]:
loops1 = torch.nonzero(src == dst).flatten().tolist()
print("Loop IDs:", loops1)

Loop IDs: [156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189]


In [15]:
nodes = torch.arange(g2.num_nodes())
loops2 = g2.edge_ids(nodes, nodes).tolist()
print("Loop IDs:", loops2)

Loop IDs: [156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189]


<p class="task" id="3"></p>

3\. Загрузите датасет `CoraFullDataset` из `dgl.data`. Этот датасет состоит из одного графа. Выведите основные характеристики графа: количество узлов, количество ребер, размерность признаков узлов, кол-во уникальных классов узлов (используйте соответствующий атрибут датасета).

Выделите подграф (`dgl.node_subgraph`), содержащий узлы, относящиеся к трем наиболее часто встречающимся классам. Выведите на экран количество узлов и ребер в полученном подграфе.



- [ ] Проверено на семинаре

In [16]:
from dgl.data import CoraFullDataset

In [17]:
dataset = CoraFullDataset()
g = dataset[0]

/root/.dgl/cora_full.zip:   0%|          | 0.00/6.15M [00:00<?, ?B/s]

Extracting file to /root/.dgl/cora_full_659794fa


In [18]:
num_nodes = g.num_nodes()
num_edges = g.num_edges()
feat_dim = g.ndata['feat'].shape[1]
labels = g.ndata['label']
num_classes = len(torch.unique(labels))

In [19]:
print("Nodes:", num_nodes)
print("Edges:", num_edges)
print("Node feature dim:", feat_dim)
print("Unique classes:", num_classes)

Nodes: 19793
Edges: 126842
Node feature dim: 8710
Unique classes: 70


In [20]:
class_counts = torch.bincount(labels)
top3 = torch.topk(class_counts, 3).indices
print("Top-3 classes:", top3.tolist())

Top-3 classes: [57, 33, 27]


In [21]:
mask = torch.isin(labels, top3)
subg = dgl.node_subgraph(g, mask)

print("Subgraph nodes:", subg.num_nodes())
print("Subgraph edges:", subg.num_edges())

Subgraph nodes: 2566
Subgraph edges: 12182


<p class="task" id="4"></p>

4\. Загрузите датасет `CoraFullDataset` из `dgl.data`.

Для каждого ребра `(u, v)` рассчитайте $L_2$ норму суммы векторов `feats[u] + feats[v]`, где `feats` - тензор атрибутов узлов в графе. Сохраните полученные величины в виде атрибута `norm` ребер.

Выделите подграф (`dgl.edge_subgraph`), содержащий ребра, для которых значение атрибута `norm` больше среднего. Выведите на экран количество узлов и ребер в полученном подграфе.

- [ ] Проверено на семинаре

In [25]:
dataset = CoraFullDataset()
g = dataset[0]
feats = g.ndata['feat']
u, v = g.edges()
E = g.num_edges()

In [26]:
batch_size = 10_000
sum_norms = 0.0
for i in range(0, E, batch_size):
    j = min(i + batch_size, E)
    s = feats[u[i:j]] + feats[v[i:j]]
    sum_norms += torch.norm(s, p=2, dim=1).sum().item()
mean_norm = sum_norms / E

In [27]:
norms = torch.empty(E, dtype=torch.float32)
for i in range(0, E, batch_size):
    j = min(i + batch_size, E)
    s = feats[u[i:j]] + feats[v[i:j]]
    norms[i:j] = torch.norm(s, p=2, dim=1)
g.edata['norm'] = norms

mask = norms > mean_norm

In [29]:
mask = norms > mean_norm
subg = dgl.edge_subgraph(g, mask, relabel_nodes=True)

print("Original graph:   nodes =", g.num_nodes(), ", edges =", g.num_edges())
print("Filtered subgraph: nodes =", subg.num_nodes(), ", edges =", subg.num_edges())

Original graph:   nodes = 19793 , edges = 126842
Filtered subgraph: nodes = 15157 , edges = 50104


<p class="task" id="5"></p>

5\. В каталоге `heterograph` находятся файлы с информацией о студентах, преподавателях и их отношениях. Постройте `dgl.heterograph`, загрузив данные из этих файлов. В полученном гетерографе должно быть:
* два типа узлов: `student` и `lecturer`
* три типа ребер: `("student", "friendship", "student")`, `("student", "coursework", "lecturer")` и `("lecturer", "collaboration", "lecturer")`
* для узлов типа `student` хранится информация о средних оценках за каждый из 4 годов обучения в виде атрибута `grades`;
* для ребер типа `lecturer` хранится информация о оценках качества работы преподавателя за каждый из 5 лет в виде атрибута `quality`.

Выведите на экран список типов узлов (`ntypes`) и список типов ребер (`etypes`). При помощи метода `number_of_nodes` выведите количество узлов каждого типа; при помощи метода `number_of_edges` выведите количество узлов каждого типа

- [ ] Проверено на семинаре

In [30]:
import pandas as pd

In [38]:
students_df = pd.read_csv('students.csv')
lecturers_df = pd.read_csv('lecturers.csv')

In [39]:
friend_df = pd.read_csv('student_friendships.csv')
course_df = pd.read_csv('student_courseworks.csv')
collab_df = pd.read_csv('lecturer_collaborations.csv')

In [43]:
student_ids = students_df['id'].unique()
student_map = {sid: i for i, sid in enumerate(student_ids)}

lecturer_ids = lecturers_df['id'].unique()
lecturer_map = {lid: i for i, lid in enumerate(lecturer_ids)}

In [45]:
data_dict = {
    ('student', 'friendship', 'student'): (
        friend_df['student1_id'].map(student_map).to_list(),
        friend_df['student2_id'].map(student_map).to_list()
    ),
    ('student', 'coursework', 'lecturer'): (
        course_df['student_id'].map(student_map).to_list(),
        course_df['lecturer_id'].map(lecturer_map).to_list()
    ),
    ('lecturer', 'collaboration', 'lecturer'): (
        collab_df['lecturer1_id'].map(lecturer_map).to_list(),
        collab_df['lecturer2_id'].map(lecturer_map).to_list()
    )
}

In [46]:
# 4. Строим гетерограф, задаём число узлов каждого типа
hg = dgl.heterograph(
    data_dict,
    num_nodes_dict={
        'student': len(student_ids),
        'lecturer': len(lecturer_ids)
    }
)


In [49]:
grades = torch.tensor(
    students_df
      .set_index('id')
      .loc[student_ids, ['year1_avg_grade','year2_avg_grade','year3_avg_grade','year4_avg_grade']]
      .values,
    dtype=torch.float32
)
hg.nodes['student'].data['grades'] = grades


In [50]:
quality = torch.tensor(
    lecturers_df
      .set_index('id')
      .loc[lecturer_ids, ['2015_quality','2016_quality','2017_quality','2018_quality','2018_quality']]
      .values,
    dtype=torch.float32
)
hg.nodes['lecturer'].data['quality'] = quality


In [51]:
print("Node types:", hg.ntypes)
print("Edge types:", hg.etypes)

# Количество узлов каждого типа
for ntype in hg.ntypes:
    print(f"Number of '{ntype}' nodes:", hg.number_of_nodes(ntype))

# Количество рёбер каждого типа
for canonical_etype in hg.canonical_etypes:
    src_type, etype, dst_type = canonical_etype
    print(f"Number of edges ({src_type}, '{etype}', {dst_type}):",
          hg.number_of_edges(canonical_etype))

Node types: ['lecturer', 'student']
Edge types: ['collaboration', 'coursework', 'friendship']
Number of 'lecturer' nodes: 5
Number of 'student' nodes: 5
Number of edges (lecturer, 'collaboration', lecturer): 4
Number of edges (student, 'coursework', lecturer): 5
Number of edges (student, 'friendship', student): 4
